In [1]:
# ============================================================
# CELL 1 — IMPORTS AND PROJECT CONFIGURATION
# ============================================================

# Manage files and directories.
from pathlib import Path

# Perform numerical calculations with vectors and matrices.
import numpy as np

# Load structured documents previously converted with Docling.
from docling_core.types.doc import DoclingDocument

# Divide documents into smaller, meaningful passages.
from docling_core.transforms.chunker.hybrid_chunker import HybridChunker

# Connect the Hugging Face tokenizer to Docling.
from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer

# Load the tokenizer associated with the BGE-M3 embedding model.
from transformers import AutoTokenizer

# Connect to the local embedding server through an OpenAI-compatible API.
from openai import OpenAI


# ============================================================
# EXISTING PROJECT PATHS — HP Z4
# ============================================================

# Directory containing Romain's RAG demonstration files.
PROJECT_ROOT = Path(
    "/home/bm/formation/RAG/cours-rag/"
    "drive-download-20261008T064617Z-1-001/rag/demo"
)

# Directory containing the JSON and Markdown exports from Exercise 1.
EXPORT_FOLDER = PROJECT_ROOT / "export_folder"

# Directory containing the local BGE-M3 model.
MODEL_PATH = Path("/home/bm/models/rag/embedder/bge-m3")


# ============================================================
# ENVIRONMENT VERIFICATION
# ============================================================

print("Project directory:", PROJECT_ROOT.is_dir())
print("Export directory:", EXPORT_FOLDER.is_dir())
print("BGE-M3 model:", MODEL_PATH.is_dir())

# Find previously exported Docling JSON files.
json_files = sorted(EXPORT_FOLDER.glob("*.json"))

print("JSON documents available:", len(json_files))

Project directory: True
Export directory: True
BGE-M3 model: True
JSON documents available: 15


In [2]:
# ============================================================
# CELL 2 — LOAD AND INSPECT FIRST DOCUMENT
# ============================================================

# 1. Identify the JSON document.
json_path = EXPORT_FOLDER / "guide_onboarding.pdf.json"

# 2. Load the structured document from JSON.
doc = DoclingDocument.load_from_json(json_path)

# 3. Convert the document to readable Markdown.
text = doc.export_to_markdown()

# 4. Display document information.
print("Document:", json_path.name)
print("Pages:", len(doc.pages))
print("Text elements:", len(doc.texts))
print("Tables:", len(doc.tables))
print("Characters:", len(text))

# 5. Display the beginning of the document.
print("\n--- DOCUMENT PREVIEW ---")
print(text[:800])

Document: guide_onboarding.pdf.json
Pages: 2
Text elements: 37
Tables: 2
Characters: 3781

--- DOCUMENT PREVIEW ---
## Guide d'accueil des nouveaux collaborateurs

Ressources humaines - version de janvier 2026

## 1. Bienvenue chez Veltis

Ce guide rassemble tout ce qu'il faut savoir pour vos premières semaines. Il ne remplace pas les échanges avec votre manager et votre parrain ou marraine : n'hésitez jamais à poser des questions. - Sophie Marchal, directrice des ressources humaines

## 2. Avant votre arrivée

## 2.1 Documents à fournir

- une copie de votre pièce d'identité ;
- votre numéro de compte bancaire pour le versement du salaire ;
- le formulaire de choix de l'assurance hospitalisation (famille ou individuelle) ;
- votre attestation de domicile si vous demandez l'indemnité vélo.

## 2.2 Votre parrain ou votre marraine

Chaque nouvelle personne est accompagnée par un parrain ou une marraine is


In [3]:
# ============================================================
# CELL 3 — LOAD BGE-M3 TOKENIZER
# ============================================================

# 1. Load the tokenizer from the local model directory.
tokenizer = AutoTokenizer.from_pretrained(
    str(MODEL_PATH),
    local_files_only=True
)

# 2. Select a short sentence from the document.
sample_text = "Chaque nouvelle personne est accompagnée pendant trois mois."

# 3. Convert the sentence into token IDs.
token_ids = tokenizer.encode(sample_text)

# 4. Convert the token IDs into readable token representations.
tokens = tokenizer.convert_ids_to_tokens(token_ids)

# 5. Display the results.
print("Original text:", sample_text)
print("Tokenizer:", tokenizer.__class__.__name__)
print("Number of tokens:", len(token_ids))
print("Token IDs:", token_ids)
print("Tokens:", tokens)

Original text: Chaque nouvelle personne est accompagnée pendant trois mois.
Tokenizer: XLMRobertaTokenizer
Number of tokens: 12
Token IDs: [0, 117967, 24164, 24932, 437, 170661, 13, 24320, 29866, 13411, 5, 2]
Tokens: ['<s>', '▁Chaque', '▁nouvelle', '▁personne', '▁est', '▁accompagné', 'e', '▁pendant', '▁trois', '▁mois', '.', '</s>']


In [4]:
# ============================================================
# CELL 4 — CONFIGURE HYBRID CHUNKER
# ============================================================

# 1. Adapt the BGE-M3 tokenizer for Docling.
docling_tokenizer = HuggingFaceTokenizer(
    tokenizer=tokenizer,
    max_tokens=512
)

# 2. Create the HybridChunker.
chunker = HybridChunker(
    tokenizer=docling_tokenizer
)

# 3. Display the configuration.
print("Tokenizer:", tokenizer.__class__.__name__)
print("Maximum tokens:", docling_tokenizer.max_tokens)
print("Chunker:", chunker.__class__.__name__)

Tokenizer: XLMRobertaTokenizer
Maximum tokens: 512
Chunker: HybridChunker


In [6]:
# ============================================================
# CELL 5 — GENERATE AND INSPECT CHUNKS
# ============================================================

# 1. Generate chunks from the document loaded in Cell 2.
chunks = list(chunker.chunk(doc))

# 2. Display the number of chunks generated.
print("Document:", json_path.name)
print("Number of chunks:", len(chunks))

# 3. Inspect the first three chunks.
for index, chunk in enumerate(chunks[:3], start=1):

    # Add contextual information, including section headings.
    contextual_text = chunker.contextualize(chunk)

    # Count tokens in the contextualized text.
    token_count = len(tokenizer.encode(contextual_text))

    # Display the chunk.
    print(f"\n===== CHUNK {index} =====")
    print("Tokens:", token_count)
    print("Text:")
    print(contextual_text[:1000])

Document: guide_onboarding.pdf.json
Number of chunks: 13

===== CHUNK 1 =====
Tokens: 21
Text:
Guide d'accueil des nouveaux collaborateurs
Ressources humaines - version de janvier 2026

===== CHUNK 2 =====
Tokens: 63
Text:
1. Bienvenue chez Veltis
Ce guide rassemble tout ce qu'il faut savoir pour vos premières semaines. Il ne remplace pas les échanges avec votre manager et votre parrain ou marraine : n'hésitez jamais à poser des questions. - Sophie Marchal, directrice des ressources humaines

===== CHUNK 3 =====
Tokens: 66
Text:
2.1 Documents à fournir
- une copie de votre pièce d'identité ;
- votre numéro de compte bancaire pour le versement du salaire ;
- le formulaire de choix de l'assurance hospitalisation (famille ou individuelle) ;
- votre attestation de domicile si vous demandez l'indemnité vélo.


In [7]:
# ============================================================
# CELL 6 — INSPECT ALL CHUNKS
# ============================================================

# Examine every chunk generated from the onboarding guide.
for index, chunk in enumerate(chunks, start=1):

    # Add section headings and contextual information.
    chunk_text = chunker.contextualize(chunk)

    # Count tokens in the contextualized passage.
    token_count = len(tokenizer.encode(chunk_text))

    # Display the complete chunk.
    print(f"\n{'=' * 20} CHUNK {index} {'=' * 20}")
    print(f"Tokens: {token_count}")
    print(chunk_text)


==================== CHUNK 1 ====================
Tokens: 21
Guide d'accueil des nouveaux collaborateurs
Ressources humaines - version de janvier 2026

==================== CHUNK 2 ====================
Tokens: 63
1. Bienvenue chez Veltis
Ce guide rassemble tout ce qu'il faut savoir pour vos premières semaines. Il ne remplace pas les échanges avec votre manager et votre parrain ou marraine : n'hésitez jamais à poser des questions. - Sophie Marchal, directrice des ressources humaines

==================== CHUNK 3 ====================
Tokens: 66
2.1 Documents à fournir
- une copie de votre pièce d'identité ;
- votre numéro de compte bancaire pour le versement du salaire ;
- le formulaire de choix de l'assurance hospitalisation (famille ou individuelle) ;
- votre attestation de domicile si vous demandez l'indemnité vélo.

==================== CHUNK 4 ====================
Tokens: 68
2.2 Votre parrain ou votre marraine
Chaque nouvelle personne est accompagnée par un parrain ou une marraine 

In [8]:
# ============================================================
# CELL 7 — PREPARE TEXTS FOR EMBEDDINGS
# ============================================================

# Convert every chunk into contextualized text.
chunk_texts = [
    chunker.contextualize(chunk)
    for chunk in chunks
]

# Verify the resulting list.
print("Total chunks:", len(chunks))
print("Texts prepared:", len(chunk_texts))

# Inspect the first passage.
print("\nFIRST TEXT:")
print(chunk_texts[0])

# Inspect the Python data types.
print("\nDATA TYPES:")
print("chunk_texts:", type(chunk_texts))
print("First element:", type(chunk_texts[0]))

Total chunks: 13
Texts prepared: 13

FIRST TEXT:
Guide d'accueil des nouveaux collaborateurs
Ressources humaines - version de janvier 2026

DATA TYPES:
chunk_texts: <class 'list'>
First element: <class 'str'>


In [10]:
# ============================================================
# CELL 8 — CHECK BGE-M3 EMBEDDING SERVER
# ============================================================

# Address of the embedding API on the Z4.
EMBEDDER_URL = "http://127.0.0.1:8001/v1"

# Create an API client.
embedding_client = OpenAI(
    base_url=EMBEDDER_URL,
    api_key="not-needed",
    timeout=5.0
)

# Ask the server which models are available.
try:
    models = embedding_client.models.list()

    print("SERVER: CONNECTED")

    for model in models.data:
        print("Available model:", model.id)

except Exception as error:
    print("SERVER: NOT AVAILABLE")
    print("Error type:", type(error).__name__)
    print("Details:", error)

SERVER: CONNECTED
Available model: bge-m3


In [11]:
# ============================================================
# CELL 9 — GENERATE FIRST EMBEDDING
# ============================================================

# 1. Select the first chunk prepared in Cell 7.
sample_chunk = chunk_texts[0]

# 2. Send the text to the BGE-M3 embedding server.
response = embedding_client.embeddings.create(
    model="bge-m3",
    input=[sample_chunk]
)

# 3. Extract the embedding from the API response.
embedding = response.data[0].embedding

# 4. Convert the embedding into a NumPy array.
embedding_vector = np.array(embedding, dtype=np.float32)

# 5. Inspect the result.
print("Source text:", sample_chunk)
print("\nPython type:", type(embedding))
print("NumPy type:", type(embedding_vector))
print("Vector shape:", embedding_vector.shape)
print("Dimensions:", embedding_vector.size)

# 6. Display only the first 10 numerical values.
print("\nFirst 10 values:")
print(embedding_vector[:10])

Source text: Guide d'accueil des nouveaux collaborateurs
Ressources humaines - version de janvier 2026

Python type: <class 'list'>
NumPy type: <class 'numpy.ndarray'>
Vector shape: (1024,)
Dimensions: 1024

First 10 values:
[-0.03482697  0.01563641 -0.04821914 -0.00428238 -0.02128824 -0.03283006
  0.02315691 -0.03251861  0.03149267  0.0201707 ]


In [12]:
# ============================================================
# CELL 10 — GENERATE THE EMBEDDING MATRIX
# ============================================================

# 1. Send all 13 contextualized chunks to BGE-M3.
response = embedding_client.embeddings.create(
    model="bge-m3",
    input=chunk_texts
)

# 2. Sort embeddings by their original input positions.
ordered_embeddings = sorted(
    response.data,
    key=lambda item: item.index
)

# 3. Convert the embeddings into a NumPy matrix.
embedding_matrix = np.array(
    [item.embedding for item in ordered_embeddings],
    dtype=np.float32
)

# 4. Check the resulting matrix.
print("Number of chunks:", len(chunk_texts))
print("Matrix shape:", embedding_matrix.shape)
print("Data type:", embedding_matrix.dtype)

# 5. Verify that each chunk has one 1,024-dimensional embedding.
assert embedding_matrix.shape == (len(chunk_texts), 1024)

print("Verification: SUCCESS")

Number of chunks: 13
Matrix shape: (13, 1024)
Data type: float32
Verification: SUCCESS


In [13]:
# ============================================================
# CELL 11 — GENERATE THE QUESTION EMBEDDING
# ============================================================

# 1. Define our first test question.
question = (
    "Je commence lundi chez Veltis : "
    "à quelle heure et où dois-je me présenter ?"
)

# 2. Generate its embedding using BGE-M3.
response = embedding_client.embeddings.create(
    model="bge-m3",
    input=[question]
)

# 3. Convert the embedding into a NumPy vector.
question_vector = np.array(
    response.data[0].embedding,
    dtype=np.float32
)

# 4. Verify the result.
print("Question:", question)
print("Question vector shape:", question_vector.shape)
print("Document matrix shape:", embedding_matrix.shape)

# 5. Confirm that both use the same embedding dimensions.
assert question_vector.shape[0] == embedding_matrix.shape[1]

print("Dimension compatibility: SUCCESS")


Question: Je commence lundi chez Veltis : à quelle heure et où dois-je me présenter ?
Question vector shape: (1024,)
Document matrix shape: (13, 1024)
Dimension compatibility: SUCCESS


In [14]:
# ============================================================
# CELL 12 — COSINE SIMILARITY AND TOP 5 RETRIEVAL
# ============================================================

# 1. Calculate the length (norm) of each document vector.
document_norms = np.linalg.norm(embedding_matrix, axis=1)

# 2. Calculate the length of the question vector.
question_norm = np.linalg.norm(question_vector)

# 3. Normalize document vectors to unit length.
normalized_documents = embedding_matrix / document_norms[:, np.newaxis]

# 4. Normalize the question vector.
normalized_question = question_vector / question_norm

# 5. Calculate cosine similarities using the dot product.
similarities = normalized_documents @ normalized_question

# 6. Rank chunks from highest to lowest similarity.
top_indices = np.argsort(similarities)[::-1][:5]

# 7. Display the five most relevant chunks.
print("QUESTION:", question)
print("\n===== TOP 5 RESULTS =====")

for rank, index in enumerate(top_indices, start=1):

    print(f"\nRANK {rank}")
    print(f"Chunk: {index + 1}")
    print(f"Similarity: {similarities[index]:.4f}")
    print("Text:")
    print(chunk_texts[index][:600])

QUESTION: Je commence lundi chez Veltis : à quelle heure et où dois-je me présenter ?

===== TOP 5 RESULTS =====

RANK 1
Chunk: 2
Similarity: 0.6002
Text:
1. Bienvenue chez Veltis
Ce guide rassemble tout ce qu'il faut savoir pour vos premières semaines. Il ne remplace pas les échanges avec votre manager et votre parrain ou marraine : n'hésitez jamais à poser des questions. - Sophie Marchal, directrice des ressources humaines

RANK 2
Chunk: 7
Similarity: 0.5837
Text:
3.3 Accès informatiques
Votre compte est créé par le service IT avant votre arrivée. Le mot de passe provisoire doit être changé dans les 24 heures, faute de quoi le compte est bloqué. En cas de problème, appelez le helpdesk au poste 2100 ou écrivez à helpdesk@veltis.example.

RANK 3
Chunk: 5
Similarity: 0.5546
Text:
3.1 Accueil
Présentez-vous à 8 h 30 à l'accueil du bâtiment A. Votre badge d'accès vous y sera remis. Il ouvre les bâtiments, le parking vélo couvert et les distributeurs de la cafétéria.

RANK 4
Chunk: 6
Simil

In [15]:
# ============================================================
# CELL 13 — VECTOR SEARCH FOR QUESTION Q02
# ============================================================

# 1. Define the second question.
question_02 = (
    "Pendant combien de temps vais-je être accompagné "
    "par un collègue à mon arrivée ?"
)

# 2. Generate the question embedding.
response_02 = embedding_client.embeddings.create(
    model="bge-m3",
    input=[question_02]
)

# 3. Convert the embedding into a NumPy vector.
vector_02 = np.array(
    response_02.data[0].embedding,
    dtype=np.float32
)

# 4. Normalize the question vector.
normalized_02 = vector_02 / np.linalg.norm(vector_02)

# 5. Compare with our normalized document matrix from Cell 12.
similarities_02 = normalized_documents @ normalized_02

# 6. Retrieve the five highest-scoring chunks.
top_02 = np.argsort(similarities_02)[::-1][:5]

# 7. Display results.
print("QUESTION Q02:", question_02)

for rank, index in enumerate(top_02, start=1):
    print(f"\nRANK {rank}")
    print("Chunk:", index + 1)
    print(f"Similarity: {similarities_02[index]:.4f}")
    print("Text:", chunk_texts[index][:500])

QUESTION Q02: Pendant combien de temps vais-je être accompagné par un collègue à mon arrivée ?

RANK 1
Chunk: 4
Similarity: 0.6118
Text: 2.2 Votre parrain ou votre marraine
Chaque nouvelle personne est accompagnée par un parrain ou une marraine issu(e) d'un autre service, pendant ses 3 mois premiers mois. Son rôle est de faciliter votre intégration : répondre aux questions pratiques, vous présenter les équipes et vous faire découvrir les ateliers.

RANK 2
Chunk: 6
Similarity: 0.5871
Text: 3.2 Programme de la journée
8 h 30, Activité = Accueil, remise du badge et du matériel. 8 h 30, Avec = Accueil RH. 9 h 30, Activité = Présentation de l'entreprise et de ses valeurs. 9 h 30, Avec = Sophie Marchal. 11 h 00, Activité = Visite des ateliers de production. 11 h 00, Avec = Votre parrain / marraine. 12 h 30, Activité = Déjeuner d'accueil à la cafétéria. 12 h 30, Avec = Votre équipe. 14 h 00, Activité = Installation du poste et des accès informatiques. 14 h 00, Avec = Service IT. 15 h 30

RANK

In [16]:
# ============================================================
# CELL 14 — VECTOR SEARCH FOR QUESTION Q03
# ============================================================

# 1. Define the third question.
question_03 = (
    "Que se passe-t-il si je ne change pas "
    "mon mot de passe provisoire ?"
)

# 2. Generate its embedding.
response_03 = embedding_client.embeddings.create(
    model="bge-m3",
    input=[question_03]
)

# 3. Convert the embedding into a NumPy vector.
vector_03 = np.array(
    response_03.data[0].embedding,
    dtype=np.float32
)

# 4. Normalize the question vector.
normalized_03 = vector_03 / np.linalg.norm(vector_03)

# 5. Compare the question with all 13 document vectors.
similarities_03 = normalized_documents @ normalized_03

# 6. Retrieve the five highest-scoring chunks.
top_03 = np.argsort(similarities_03)[::-1][:5]

# 7. Display the results.
print("QUESTION Q03:", question_03)

for rank, index in enumerate(top_03, start=1):
    print(f"\nRANK {rank}")
    print("Chunk:", index + 1)
    print(f"Similarity: {similarities_03[index]:.4f}")
    print("Text:", chunk_texts[index][:500])

QUESTION Q03: Que se passe-t-il si je ne change pas mon mot de passe provisoire ?

RANK 1
Chunk: 7
Similarity: 0.6464
Text: 3.3 Accès informatiques
Votre compte est créé par le service IT avant votre arrivée. Le mot de passe provisoire doit être changé dans les 24 heures, faute de quoi le compte est bloqué. En cas de problème, appelez le helpdesk au poste 2100 ou écrivez à helpdesk@veltis.example.

RANK 2
Chunk: 8
Similarity: 0.4340
Text: 4.1 Formation sécurité
Une formation sécurité de 3 heures est obligatoire avant tout accès à l'atelier, quel que soit votre poste. Elle a lieu le mardi ou le jeudi de votre première semaine. Les chaussures de sécurité et les lunettes de protection vous sont fournies.

RANK 3
Chunk: 13
Similarity: 0.4254
Text: 6. Contacts utiles
Question RH, contrat, congés, Contact = Service RH. Question RH, contrat, congés, Poste = 2000. Informatique, accès, badge, Contact = Helpdesk IT. Informatique, accès, badge, Poste = 2100. Sécurité, accident, Contact = Responsa

In [17]:
# ============================================================
# CELL 15 — REUSABLE VECTOR SEARCH FUNCTION
# ============================================================

def search_chunks(question_text, top_k=5):

    # 1. Generate an embedding for the question.
    response = embedding_client.embeddings.create(
        model="bge-m3",
        input=[question_text]
    )

    # 2. Convert the response into a NumPy vector.
    query_vector = np.array(
        response.data[0].embedding,
        dtype=np.float32
    )

    # 3. Normalize the question vector.
    query_norm = np.linalg.norm(query_vector)

    if query_norm == 0:
        raise ValueError("Question embedding has zero length.")

    normalized_query = query_vector / query_norm

    # 4. Calculate cosine similarity with all document chunks.
    scores = normalized_documents @ normalized_query

    # 5. Select the highest-scoring chunks.
    top_indices = np.argsort(scores)[::-1][:top_k]

    # 6. Return the results with their original chunk numbers.
    results = []

    for index in top_indices:
        results.append({
            "chunk_id": int(index) + 1,
            "score": float(scores[index]),
            "text": chunk_texts[index]
        })

    return results

In [18]:
# ============================================================
# CELL 16 — VALIDATE REUSABLE VECTOR SEARCH
# ============================================================

# Store the three questions and their expected chunks.
test_questions = {
    "Q01": {
        "question": "Je commence lundi chez Veltis : à quelle heure et où dois-je me présenter ?",
        "expected_chunk": 5
    },
    "Q02": {
        "question": "Pendant combien de temps vais-je être accompagné par un collègue à mon arrivée ?",
        "expected_chunk": 4
    },
    "Q03": {
        "question": "Que se passe-t-il si je ne change pas mon mot de passe provisoire ?",
        "expected_chunk": 7
    }
}

# Evaluate the reusable function on each question.
for question_id, test in test_questions.items():

    # Retrieve the five most similar chunks.
    results = search_chunks(test["question"], top_k=5)

    # Extract their chunk identifiers.
    retrieved_ids = [item["chunk_id"] for item in results]

    # Identify the position of the expected chunk.
    expected = test["expected_chunk"]

    rank = (
        retrieved_ids.index(expected) + 1
        if expected in retrieved_ids
        else None
    )

    # Display evaluation results.
    print(f"\n{question_id}")
    print("Expected chunk:", expected)
    print("Retrieved chunks:", retrieved_ids)
    print("Correct chunk rank:", rank)
    print("Hit@5:", rank is not None)
    


Q01
Expected chunk: 5
Retrieved chunks: [2, 7, 5, 6, 12]
Correct chunk rank: 3
Hit@5: True

Q02
Expected chunk: 4
Retrieved chunks: [4, 6, 9, 8, 5]
Correct chunk rank: 1
Hit@5: True

Q03
Expected chunk: 7
Retrieved chunks: [7, 8, 13, 10, 4]
Correct chunk rank: 1
Hit@5: True


In [19]:
# ============================================================
# CELL 17 — CHUNK ALL VELTIS DOCUMENTS
# ============================================================

# Create an empty list for the complete corpus.
all_chunks = []

# Process each JSON document found in Cell 1.
for file in json_files:

    # Reload the structured Docling document.
    document = DoclingDocument.load_from_json(file)

    # Generate chunks using our existing HybridChunker.
    document_chunks = list(chunker.chunk(document))

    # Store each chunk with its source document.
    for chunk in document_chunks:

        all_chunks.append({
            "source": file.name.removesuffix(".json"),
            "text": chunker.contextualize(chunk)
        })

    # Display the number of chunks generated for this document.
    print(f"{file.name}: {len(document_chunks)} chunks")

# Display the total.
print("\nTotal documents:", len(json_files))
print("Total chunks:", len(all_chunks))

grille_salariale_2026.pdf.json: 4 chunks
guide_onboarding.pdf.json: 13 chunks
intranet_actualites.html.json: 5 chunks
intranet_faq_rh.html.json: 8 chunks
newsletter_le_fil_2026-02.pdf.json: 6 chunks
organigramme_et_processus_production.pdf.json: 1 chunks
plan_strategique_2026-2028.pptx.json: 1 chunks
politique_teletravail_2023.docx.json: 6 chunks
politique_teletravail_2025.docx.json: 7 chunks
politique_teletravail_2025.pdf.json: 7 chunks
presentation_veltis.pdf.json: 9 chunks
pv_conseil_administration_2026-03-12_scan.pdf.json: 6 chunks
rapport_annuel_2025.pdf.json: 12 chunks

Total documents: 13
Total chunks: 85


In [23]:
# ============================================================
# CELL 18B — INCREASE TOKEN LIMIT AND REGENERATE CHUNKS
# ============================================================

# Increase the maximum from 512 to 544 tokens.
docling_tokenizer = HuggingFaceTokenizer(
    tokenizer=tokenizer,
    max_tokens=544
)

# Recreate the chunker with the new limit.
chunker = HybridChunker(tokenizer=docling_tokenizer)

# Clear the previous chunk collection.
all_chunks = []

# Regenerate chunks for all 13 documents.
for file in json_files:

    document = DoclingDocument.load_from_json(file)

    for chunk in chunker.chunk(document):
        all_chunks.append({
            "source": file.name.removesuffix(".json"),
            "text": chunker.contextualize(chunk)
        })

# Recalculate token counts.
all_chunk_texts = [item["text"] for item in all_chunks]

token_counts = [
    len(tokenizer.encode(text))
    for text in all_chunk_texts
]

print("Total chunks:", len(all_chunks))
print("Maximum tokens:", max(token_counts))
print("Chunks exceeding 544:", sum(n > 544 for n in token_counts))

Total chunks: 84
Maximum tokens: 553
Chunks exceeding 544: 2


In [24]:
# ============================================================
# CELL 18C — VALIDATE CHUNKS FOR EMBEDDING
# ============================================================

# Our practical embedding input limit.
EMBEDDING_TOKEN_LIMIT = 576

# Count chunks exceeding this limit.
oversized = [
    count for count in token_counts
    if count > EMBEDDING_TOKEN_LIMIT
]

# Count empty passages.
empty = [
    text for text in all_chunk_texts
    if not text.strip()
]

# Display validation.
print("Total chunks:", len(all_chunk_texts))
print("Maximum tokens:", max(token_counts))
print("Empty chunks:", len(empty))
print("Chunks exceeding 576:", len(oversized))

# Stop only if an actual problem remains.
assert len(empty) == 0, "Empty chunks detected"
assert len(oversized) == 0, "Embedding input limit exceeded"

print("Validation: SUCCESS")

Total chunks: 84
Maximum tokens: 553
Empty chunks: 0
Chunks exceeding 576: 0
Validation: SUCCESS


In [25]:
# ============================================================
# CELL 19 — GENERATE EMBEDDINGS FOR ALL DOCUMENTS
# ============================================================

# 1. Send the 84 contextualized chunks to BGE-M3.
response = embedding_client.embeddings.create(
    model="bge-m3",
    input=all_chunk_texts
)

# 2. Sort embeddings by their original input positions.
ordered_embeddings = sorted(
    response.data,
    key=lambda item: item.index
)

# 3. Convert the returned vectors into a NumPy matrix.
all_embeddings = np.array(
    [item.embedding for item in ordered_embeddings],
    dtype=np.float32
)

# 4. Verify the matrix dimensions.
print("Documents:", len(json_files))
print("Chunks:", len(all_chunk_texts))
print("Embedding matrix:", all_embeddings.shape)
print("Data type:", all_embeddings.dtype)

# 5. Check that every chunk has one embedding.
assert all_embeddings.shape == (len(all_chunk_texts), 1024)

# 6. Check that all values are valid numbers.
assert np.isfinite(all_embeddings).all()

print("Embedding validation: SUCCESS")

Documents: 13
Chunks: 84
Embedding matrix: (84, 1024)
Data type: float32
Embedding validation: SUCCESS


In [26]:
# ============================================================
# CELL 20 — NORMALIZE THE COMPLETE EMBEDDING MATRIX
# ============================================================

# 1. Calculate the Euclidean norm of each embedding.
vector_norms = np.linalg.norm(all_embeddings, axis=1)

# 2. Verify that no vector has zero length.
assert np.all(vector_norms > 0), "Zero-length embedding detected!"

# 3. Normalize each embedding to unit length.
normalized_all_embeddings = all_embeddings / vector_norms[:, np.newaxis]

# 4. Verify the normalized matrix.
normalized_norms = np.linalg.norm(
    normalized_all_embeddings,
    axis=1
)

print("Original matrix:", all_embeddings.shape)
print("Normalized matrix:", normalized_all_embeddings.shape)

print("\nNORMALIZATION CHECK")
print("Minimum vector norm:", round(normalized_norms.min(), 6))
print("Maximum vector norm:", round(normalized_norms.max(), 6))

# 5. Verify that all vector lengths are approximately 1.
assert np.allclose(normalized_norms, 1.0, atol=1e-5)

print("Normalization: SUCCESS")

Original matrix: (84, 1024)
Normalized matrix: (84, 1024)

NORMALIZATION CHECK
Minimum vector norm: 1.0
Maximum vector norm: 1.0
Normalization: SUCCESS


In [27]:
# ============================================================
# CELL 21 — VECTOR SEARCH ACROSS THE COMPLETE CORPUS
# ============================================================

# 1. Define our test question.
query = (
    "Je commence lundi chez Veltis : "
    "à quelle heure et où dois-je me présenter ?"
)

# 2. Generate an embedding for the question.
response = embedding_client.embeddings.create(
    model="bge-m3",
    input=[query]
)

# 3. Convert the embedding to a NumPy vector.
query_vector = np.array(
    response.data[0].embedding,
    dtype=np.float32
)

# 4. Normalize the question vector.
query_norm = np.linalg.norm(query_vector)

if query_norm == 0:
    raise ValueError("Question embedding has zero length.")

normalized_query = query_vector / query_norm

# 5. Calculate cosine similarities against all 84 chunks.
scores = normalized_all_embeddings @ normalized_query

# 6. Select the five highest-scoring chunks.
top_indices = np.argsort(scores)[::-1][:5]

# 7. Display the results with their original sources.
print("QUESTION:", query)
print("\n===== TOP 5 RESULTS — COMPLETE CORPUS =====")

for rank, index in enumerate(top_indices, start=1):

    print(f"\nRANK {rank}")
    print("Source:", all_chunks[index]["source"])
    print("Global chunk index:", index)
    print(f"Similarity: {scores[index]:.4f}")
    print("Text:")
    print(all_chunks[index]["text"][:500])

QUESTION: Je commence lundi chez Veltis : à quelle heure et où dois-je me présenter ?

===== TOP 5 RESULTS — COMPLETE CORPUS =====

RANK 1
Source: guide_onboarding.pdf
Global chunk index: 5
Similarity: 0.6002
Text:
1. Bienvenue chez Veltis
Ce guide rassemble tout ce qu'il faut savoir pour vos premières semaines. Il ne remplace pas les échanges avec votre manager et votre parrain ou marraine : n'hésitez jamais à poser des questions. - Sophie Marchal, directrice des ressources humaines

RANK 2
Source: guide_onboarding.pdf
Global chunk index: 10
Similarity: 0.5838
Text:
3.3 Accès informatiques
Votre compte est créé par le service IT avant votre arrivée. Le mot de passe provisoire doit être changé dans les 24 heures, faute de quoi le compte est bloqué. En cas de problème, appelez le helpdesk au poste 2100 ou écrivez à helpdesk@veltis.example.

RANK 3
Source: presentation_veltis.pdf
Global chunk index: 58
Similarity: 0.5739
Text:
Présentation de l'entreprise
Veltis SA - édition 2026

RANK 4

In [28]:
# ============================================================
# CELL 22 — REUSABLE SEARCH ACROSS ALL 13 DOCUMENTS
# ============================================================

def search_corpus(question_text, top_k=5):

    # Generate an embedding for the question.
    response = embedding_client.embeddings.create(
        model="bge-m3",
        input=[question_text]
    )

    # Convert the embedding into a NumPy vector.
    query_vector = np.array(
        response.data[0].embedding,
        dtype=np.float32
    )

    # Calculate the vector norm.
    query_norm = np.linalg.norm(query_vector)

    # Prevent division by zero.
    if query_norm == 0:
        raise ValueError("Question embedding has zero norm.")

    # Normalize the question vector.
    normalized_query = query_vector / query_norm

    # Calculate similarity against all 84 chunks.
    scores = normalized_all_embeddings @ normalized_query

    # Retrieve indices of the highest scores.
    top_indices = np.argsort(scores)[::-1][:top_k]

    # Build results while preserving document metadata.
    results = []

    for index in top_indices:
        results.append({
            "global_index": int(index),
            "source": all_chunks[index]["source"],
            "score": float(scores[index]),
            "text": all_chunks[index]["text"]
        })

    return results

In [29]:
# ============================================================
# CELL 23 — EVALUATE RETRIEVAL ACROSS THE COMPLETE CORPUS
# ============================================================

# Define our test questions and expected answer passages.
test_cases = {
    "Q01": {
        "question": "Je commence lundi chez Veltis : à quelle heure et où dois-je me présenter ?",
        "expected_source": "guide_onboarding.pdf",
        "expected_text": "Présentez-vous à 8 h 30"
    },
    "Q02": {
        "question": "Pendant combien de temps vais-je être accompagné par un collègue à mon arrivée ?",
        "expected_source": "guide_onboarding.pdf",
        "expected_text": "pendant ses 3 mois premiers mois"
    },
    "Q03": {
        "question": "Que se passe-t-il si je ne change pas mon mot de passe provisoire ?",
        "expected_source": "guide_onboarding.pdf",
        "expected_text": "le compte est bloqué"
    }
}

# Store evaluation results.
evaluation_results = {}

# Evaluate each question.
for question_id, test in test_cases.items():

    # Search the complete corpus.
    results = search_corpus(test["question"], top_k=5)

    # Find the rank of the passage containing the expected answer.
    correct_rank = None

    for rank, result in enumerate(results, start=1):

        source_matches = result["source"] == test["expected_source"]

        text_matches = (
            test["expected_text"].lower()
            in result["text"].lower()
        )

        if source_matches and text_matches:
            correct_rank = rank
            break

    # Save the result.
    evaluation_results[question_id] = correct_rank

    # Display the retrieved sources and scores.
    print(f"\n===== {question_id} =====")

    for rank, result in enumerate(results, start=1):
        print(
            f"{rank}. {result['source']} "
            f"| Score: {result['score']:.4f}"
        )

    print("Correct answer rank:", correct_rank)
    print("Hit@5:", correct_rank is not None)


# ============================================================
# EVALUATION SUMMARY
# ============================================================

total = len(evaluation_results)

hit_at_1 = sum(rank == 1 for rank in evaluation_results.values())

hit_at_5 = sum(rank is not None for rank in evaluation_results.values())

mrr = sum(
    1 / rank if rank is not None else 0
    for rank in evaluation_results.values()
) / total

print("\n===== EVALUATION SUMMARY =====")
print("Questions:", total)
print(f"Hit@1: {hit_at_1}/{total}")
print(f"Hit@5: {hit_at_5}/{total}")
print(f"MRR@5: {mrr:.3f}")


===== Q01 =====
1. guide_onboarding.pdf | Score: 0.6002
2. guide_onboarding.pdf | Score: 0.5838
3. presentation_veltis.pdf | Score: 0.5739
4. newsletter_le_fil_2026-02.pdf | Score: 0.5669
5. guide_onboarding.pdf | Score: 0.5547
Correct answer rank: 5
Hit@5: True

===== Q02 =====
1. guide_onboarding.pdf | Score: 0.6118
2. guide_onboarding.pdf | Score: 0.5872
3. guide_onboarding.pdf | Score: 0.5863
4. guide_onboarding.pdf | Score: 0.5822
5. politique_teletravail_2025.pdf | Score: 0.5692
Correct answer rank: 1
Hit@5: True

===== Q03 =====
1. guide_onboarding.pdf | Score: 0.6463
2. guide_onboarding.pdf | Score: 0.4339
3. politique_teletravail_2025.pdf | Score: 0.4315
4. guide_onboarding.pdf | Score: 0.4253
5. grille_salariale_2026.pdf | Score: 0.4110
Correct answer rank: 1
Hit@5: True

===== EVALUATION SUMMARY =====
Questions: 3
Hit@1: 2/3
Hit@5: 3/3
MRR@5: 0.733


In [30]:
# ============================================================
# CELL 24 — DIVIDEND QUESTION: VECTOR RETRIEVAL
# ============================================================

# 1. Define the question discussed during Romain's lecture.
dividend_question = (
    "Quel dividende sera proposé aux actionnaires ?"
)

# 2. Search the complete corpus.
dividend_results = search_corpus(
    dividend_question,
    top_k=5
)

# 3. Display the retrieved passages.
print("QUESTION:", dividend_question)

for rank, result in enumerate(dividend_results, start=1):

    print(f"\n===== RANK {rank} =====")

    print("Source:", result["source"])
    print("Global index:", result["global_index"])
    print(f"Similarity: {result['score']:.4f}")

    print("Text:")
    print(result["text"][:700])

QUESTION: Quel dividende sera proposé aux actionnaires ?

===== RANK 1 =====
Source: pv_conseil_administration_2026-03-12_scan.pdf
Global index: 69
Similarity: 0.6330
Text:
2. Arrêt des comptes de 1'exercice 2025
Le directeur financier présente le rapport de gestion et les comptes annuels. Après échange, le conseil arrête les comptes à l'unanimité et décide de proposer à l'assemblée générale la distribution d'un dividende de 0,80€ par action.

===== RANK 2 =====
Source: rapport_annuel_2025.pdf
Global index: 76
Similarity: 0.4747
Text:
3. Chiffres clés
Chiffre d'affaires (M€), 2024 = 31,2. Chiffre d'affaires (M€), 2025 = 38,4. Chiffre d'affaires (M€), Évolution = +23,1 %. EBITDA (M€), 2024 = 3,1. EBITDA (M€), 2025 = 4,6. EBITDA (M€), Évolution = +48,4 %. Résultat net (M€), 2024 = 1,2. Résultat net (M€), 2025 = 2,1. Résultat net (M€), Évolution = +75,0 %. Vélos vendus, 2024 = 8 020. Vélos vendus, 2025 = 9 850. Vélos vendus, Évolution = +22,8 %. Effectif au 31 décembre, 2024 = 164. Effect

In [31]:
# ============================================================
# CELL 25 — DISPLAY QUESTION AND SOURCE EVIDENCE
# ============================================================

def show_evidence(question_text, top_k=3):

    # Retrieve relevant passages from the complete corpus.
    results = search_corpus(question_text, top_k=top_k)

    # Display the original question.
    print("QUESTION:")
    print(question_text)

    # Display the passages supporting a possible answer.
    print("\nSOURCE EVIDENCE:")

    for rank, result in enumerate(results, start=1):

        print(f"\n--- Evidence {rank} ---")

        # Identify the original document.
        print("Document:", result["source"])

        # Identify the passage within our collection.
        print("Global chunk index:", result["global_index"])

        # Display semantic similarity.
        print(f"Similarity: {result['score']:.4f}")

        # Display the complete passage rather than truncating it.
        print("Extracted text:")
        print(result["text"])


# Test with the dividend question.
show_evidence(
    "Quel dividende sera proposé aux actionnaires ?",
    top_k=3
)

QUESTION:
Quel dividende sera proposé aux actionnaires ?

SOURCE EVIDENCE:

--- Evidence 1 ---
Document: pv_conseil_administration_2026-03-12_scan.pdf
Global chunk index: 69
Similarity: 0.6330
Extracted text:
2. Arrêt des comptes de 1'exercice 2025
Le directeur financier présente le rapport de gestion et les comptes annuels. Après échange, le conseil arrête les comptes à l'unanimité et décide de proposer à l'assemblée générale la distribution d'un dividende de 0,80€ par action.

--- Evidence 2 ---
Document: rapport_annuel_2025.pdf
Global chunk index: 76
Similarity: 0.4747
Extracted text:
3. Chiffres clés
Chiffre d'affaires (M€), 2024 = 31,2. Chiffre d'affaires (M€), 2025 = 38,4. Chiffre d'affaires (M€), Évolution = +23,1 %. EBITDA (M€), 2024 = 3,1. EBITDA (M€), 2025 = 4,6. EBITDA (M€), Évolution = +48,4 %. Résultat net (M€), 2024 = 1,2. Résultat net (M€), 2025 = 2,1. Résultat net (M€), Évolution = +75,0 %. Vélos vendus, 2024 = 8 020. Vélos vendus, 2025 = 9 850. Vélos vendus, Évolution 

In [32]:
# ============================================================
# CELL 26 — SAVE EMBEDDINGS AND SOURCE METADATA
# ============================================================

import json

# Define output paths.
embeddings_path = EXPORT_FOLDER / "veltis_embeddings.npz"
metadata_path = EXPORT_FOLDER / "veltis_chunks.json"

# Save the embedding matrix in compressed NumPy format.
np.savez_compressed(
    embeddings_path,
    embeddings=all_embeddings
)

# Save the associated source information and texts.
with open(metadata_path, "w", encoding="utf-8") as file:
    json.dump(
        all_chunks,
        file,
        ensure_ascii=False,
        indent=2
    )

# Verify that the files exist.
assert embeddings_path.is_file()
assert metadata_path.is_file()

print("Embeddings saved:", embeddings_path.name)
print("Metadata saved:", metadata_path.name)
print("Vectors:", all_embeddings.shape[0])
print("Dimensions:", all_embeddings.shape[1])
print("Metadata records:", len(all_chunks))

assert len(all_chunks) == all_embeddings.shape[0]

print("Persistence: SUCCESS")

Embeddings saved: veltis_embeddings.npz
Metadata saved: veltis_chunks.json
Vectors: 84
Dimensions: 1024
Metadata records: 84
Persistence: SUCCESS


In [33]:
# ============================================================
# CELL 27 — RELOAD AND VERIFY PERSISTED DATA
# ============================================================

# 1. Reload the saved NumPy embedding matrix.
with np.load(embeddings_path) as saved:
    loaded_embeddings = saved["embeddings"]

# 2. Reload the corresponding source metadata.
with open(metadata_path, "r", encoding="utf-8") as file:
    loaded_chunks = json.load(file)

# 3. Compare the loaded data with the original data.
assert loaded_embeddings.shape == all_embeddings.shape
assert np.array_equal(loaded_embeddings, all_embeddings)
assert loaded_chunks == all_chunks

# 4. Verify that each vector still corresponds to its source.
print("Loaded embedding matrix:", loaded_embeddings.shape)
print("Loaded metadata records:", len(loaded_chunks))

print("\nExample — vector 69:")
print("Source:", loaded_chunks[69]["source"])
print("Text:", loaded_chunks[69]["text"][:200])

print("\nPersistence verification: SUCCESS")

Loaded embedding matrix: (84, 1024)
Loaded metadata records: 84

Example — vector 69:
Source: pv_conseil_administration_2026-03-12_scan.pdf
Text: 2. Arrêt des comptes de 1'exercice 2025
Le directeur financier présente le rapport de gestion et les comptes annuels. Après échange, le conseil arrête les comptes à l'unanimité et décide de proposer à

Persistence verification: SUCCESS
